# 📱 Spot Republic · Promo TikTok con IA

Convierte tus clips DJI de la carpeta **Server** en un TikTok vertical que **explica tu server con tu propia voz**:

1. **Transcripción:** Whisper large-v3 transcribe lo que dices en cada clip, palabra por palabra.
2. **IA editora:** un modelo local (Qwen3.8-27B en tu GPU, sin API key) arma el guion. Elige tus mejores frases (qué es el server, por qué entrar y cómo unirse), pone un gancho al inicio, quita repeticiones y tomas falsas, y decide qué palabras van en la pared.
3. **Cortes en tus frases** (no al ritmo de la música), con punch-in en los saltos. Tu voz se limpia y la música queda de fondo y baja cuando hablas.
4. **Textos en la pared:** palabras clave en perspectiva, pegadas a la pared detrás de ti. Siguen el movimiento de la cámara y quedan *detrás* de tu cuerpo (segmentación YOLO). En primeros planos sin pared visible van arriba con sombra.
5. **Subtítulos estilo TikTok** con la palabra activa resaltada.
6. **Vertical 1080×1920** centrado en ti, de 40 a 120 s (75 s por defecto).

**Cómo usarlo:** Entorno de ejecución → Cambiar tipo → **H100**. Luego **Entorno de ejecución → Ejecutar todo**. La 1.ª vez descarga el modelo (~56 GB).
Resultado en `Server/Edicion_IA/`: `spot_republic_promo_tiktok.mp4`, `guion_promo_tiktok.md` y `transcripcion.txt`.
Para cambiar algo, edita `PLAN` (celda 5) y vuelve a correr las celdas 6–9.

In [ ]:
#@title 1 · Instalar herramientas (Whisper, YOLO, vLLM) y montar Google Drive
#@markdown Usa **Entorno de ejecución → Cambiar tipo → GPU H100 (o A100 80 GB)**. Tarda ~3-5 min.
import subprocess, sys, os
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'faster-whisper', 'librosa', 'ultralytics', 'open_clip_torch', 'anthropic>=0.70'], check=True)
    # vLLM va en un entorno aparte para no romper el torch de Colab (usado por CLIP/YOLO/Whisper)
    if not os.path.exists('/content/vllm_env/bin/vllm'):
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'], check=True)
        subprocess.run(['uv', 'venv', '-q', '--python', '3.12', '/content/vllm_env'], check=True)
        subprocess.run(['uv', 'pip', 'install', '-q', '--python', '/content/vllm_env/bin/python', '-U', 'vllm'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'openai'], check=True)
    from google.colab import drive
    drive.mount('/content/drive')
try:
    print(subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout or 'Sin GPU (irá más lento)')
except FileNotFoundError:
    print('Sin GPU (irá más lento)')

In [ ]:
#@title 2 · Configuración de la promo TikTok
import os, glob, json, math, random, re, shutil, subprocess, base64, gc, time
import numpy as np, cv2

DRIVE_FOLDER   = "/content/drive/MyDrive/Server"  #@param {type:"string"}
SOUNDTRACK     = "spot_republic_soundtrack_master.mp4"  #@param {type:"string"}
NOMBRE_SERVER  = "Spot Republic"  #@param {type:"string"}
DURACION_OBJETIVO_S = 75  #@param {type:"slider", min:40, max:120, step:5}
#@markdown Lo que dices en tus clips es la base del video. La música va de fondo y baja cuando hablas.
VOLUMEN_MUSICA = 0.14  #@param {type:"slider", min:0, max:0.5, step:0.01}
#@markdown Texto final (IP, Discord, "link en la bio"…). Vacío = lo propone la IA a partir de lo que dices.
TEXTO_CTA      = ""  #@param {type:"string"}
IDIOMA         = "es"  #@param ["es", "en"]
GRADE          = "natural"  #@param ["natural", "teal_orange", "warm_film", "ninguno"]
#@markdown **Motor de IA del editor** · `local_qwen` = Qwen3.8-27B en tu GPU (H100/A100 80 GB) · `claude` = API de Anthropic (secreto `ANTHROPIC_API_KEY`) · `simple` = sin IA (orden cronológico).
MOTOR_IA       = "local_qwen"  #@param ["local_qwen", "claude", "simple"]
MODELO_LOCAL   = "Qwen/Qwen3.8-27B"  #@param {type:"string"}
FPS = 30
OUT_W, OUT_H = 1080, 1920
random.seed(7); np.random.seed(7)

def find_folder(start="/content/drive/MyDrive", max_depth=3):
    if os.path.exists(os.path.join(DRIVE_FOLDER, SOUNDTRACK)):
        return DRIVE_FOLDER
    base = start.rstrip('/').count('/')
    for root, dirs, files in os.walk(start):
        if SOUNDTRACK in files:
            return root
        if root.count('/') - base >= max_depth:
            dirs[:] = []
    raise FileNotFoundError(f"No encuentro {SOUNDTRACK} en Drive. Ajusta DRIVE_FOLDER.")

FOLDER = find_folder()
WORK = os.environ.get("SPOT_WORK", "/content/work_tiktok")
OUT_DIR = os.path.join(FOLDER, "Edicion_IA")
for d in (WORK, f"{WORK}/proxy", f"{WORK}/frames", f"{WORK}/audio", OUT_DIR):
    os.makedirs(d, exist_ok=True)
print("Carpeta:", FOLDER, "\nSalida:", OUT_DIR)

In [ ]:
#@title 3 · Inventario, quitar duplicados y crear proxies (hasta 1440p para el recorte vertical)
def run(cmd, **kw):
    r = subprocess.run(cmd, capture_output=True, text=True, **kw)
    if r.returncode != 0:
        raise RuntimeError(f"Falló: {' '.join(map(str, cmd))[:300]}\n{r.stderr[-1500:]}")
    return r

def probe(path):
    r = run(['ffprobe', '-v', 'error', '-print_format', 'json', '-show_streams', '-show_format', path])
    j = json.loads(r.stdout)
    v = next((s for s in j['streams'] if s['codec_type'] == 'video'), None)
    a = next((s for s in j['streams'] if s['codec_type'] == 'audio'), None)
    return {'duration': float(j['format'].get('duration', 0)), 'video': v, 'audio': a}

def pick_encoder():
    try:
        run(['ffmpeg', '-v', 'error', '-f', 'lavfi', '-i', 'color=s=256x256:d=0.2', '-c:v', 'h264_nvenc', '-f', 'null', '-'])
        return ['-c:v', 'h264_nvenc', '-preset', 'p6', '-cq', '19', '-b:v', '0']
    except Exception:
        return ['-c:v', 'libx264', '-preset', 'medium', '-crf', '18']
VENC = pick_encoder()
print("Encoder:", VENC[1])

all_mp4 = sorted(glob.glob(os.path.join(FOLDER, '*.mp4')), key=lambda p: (re.sub(r' \(\d+\)(?=\.mp4$)', '', p), bool(re.search(r' \(\d+\)\.mp4$', p)), p))  # originales antes que 'nombre (1).mp4'
clips, seen = [], set()
for p in all_mp4:
    name = os.path.basename(p)
    if name == SOUNDTRACK or name.startswith('spot_republic') or os.path.getsize(p) == 0:
        continue
    info = probe(p)
    if info['video'] is None or info['duration'] < 1.5:
        continue
    key = (os.path.getsize(p), round(info['duration'], 2))  # copias subidas dos veces
    if key in seen:
        print("  duplicado, omitido:", name); continue
    seen.add(key)
    clips.append({'src': p, 'name': name, **info})
print(f"{len(clips)} clips únicos, {sum(c['duration'] for c in clips)/60:.1f} min de material")

HDR_TRC = {'arib-std-b67', 'smpte2084'}
def make_proxy(c, i):
    out = f"{WORK}/proxy/clip{i:02d}.mp4"
    c['proxy'] = out
    if os.path.exists(out) and os.path.getsize(out) > 0:
        return
    v = c['video']
    short = min(1440, min(v['width'], v['height']))         # más resolución = recorte 9:16 más nítido
    short -= short % 2
    scale = f"scale='if(gt(iw,ih),-2,{short})':'if(gt(iw,ih),{short},-2)':flags=lanczos"
    chains = []
    if v.get('color_transfer') in HDR_TRC:   # DJI HLG/HDR -> SDR Rec.709
        chains.append("zscale=t=linear:npl=100,format=gbrpf32le,zscale=p=bt709,"
                      "tonemap=hable:desat=0,zscale=t=bt709:m=bt709:r=tv,format=yuv420p," + scale)
    chains.append(scale + ",format=yuv420p")
    for vf in chains:
        try:
            run(['ffmpeg', '-y', '-v', 'error', '-i', c['src'], '-vf', f"{vf},fps={FPS}",
                 *VENC, '-g', str(FPS), '-c:a', 'aac', '-b:a', '192k', '-ac', '2', '-ar', '48000', out])
            return
        except RuntimeError as e:
            print("  reintentando sin tonemap:", str(e)[-200:])
    raise RuntimeError("No se pudo crear proxy de " + c['name'])

for i, c in enumerate(clips):
    print(f"proxy {i+1}/{len(clips)}: {c['name']}  ({c['duration']:.0f}s, {c['video']['width']}x{c['video']['height']}, trc={c['video'].get('color_transfer')})")
    make_proxy(c, i)
    cap = cv2.VideoCapture(c['proxy'])
    c['W'], c['H'] = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    c['nframes'] = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)); cap.release()
    c['has_audio'] = c['audio'] is not None

In [ ]:
#@title 4 · IA: transcribir lo que dices en cada clip (Whisper, palabra por palabra)
def preload_cuda12_libs():
    """faster-whisper (CTranslate2) necesita cuBLAS 12 y cuDNN 9; Colab puede traer otra versión de CUDA."""
    import ctypes, sys, site
    pat = ('libcublasLt.so.12', 'libcublas.so.12', 'libcudnn*.so.9')
    roots = site.getsitepackages() + [site.getusersitepackages()]
    find = lambda: [f for r in roots for q in pat for f in glob.glob(os.path.join(r, 'nvidia', '**', q), recursive=True)]
    if not any('libcublas.so.12' in f for f in find()):
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'nvidia-cublas-cu12', 'nvidia-cudnn-cu12==9.*'], check=True)
    libs = sorted(set(find()), key=lambda f: ('cublasLt' not in f, 'cublas' not in f, f))
    for _ in range(3):
        pending = []
        for f in libs:
            try: ctypes.CDLL(f, mode=ctypes.RTLD_GLOBAL)
            except OSError: pending.append(f)
        libs = pending
        if not libs: break

def split_phrases(words, max_len=7.0):
    """Corta en frases: puntuación, pausas > 0.45 s o > max_len segundos."""
    out, cur = [], []
    for w in words:
        if cur and (w[0] - cur[-1][1] > 0.45 or w[1] - cur[0][0] > max_len):
            out.append(cur); cur = []
        cur.append(w)
        if re.search(r'[.!?¿¡]$', w[2]) and w[1] - cur[0][0] > 1.2:
            out.append(cur); cur = []
    if cur: out.append(cur)
    return out

from faster_whisper import WhisperModel
try:
    preload_cuda12_libs()
except Exception as e:
    print("No pude preparar CUDA 12 para Whisper:", e)
def load_whisper(gpu=True):
    if gpu:
        try: return WhisperModel('large-v3', device='cuda', compute_type='float16')
        except Exception as e: print("Whisper en GPU no disponible:", str(e)[:200])
    print("Usando Whisper en CPU (más lento)")
    return WhisperModel('medium', device='cpu', compute_type='int8', cpu_threads=os.cpu_count())
wm = load_whisper(True)

def transcribe(path):
    segs, _ = wm.transcribe(path, language=IDIOMA, word_timestamps=True, vad_filter=True,
                            initial_prompt=f"{NOMBRE_SERVER}, servidor, server.")
    return [(w.start, w.end, w.word.strip()) for sg in segs for w in (sg.words or []) if w.word.strip()]

PHRASES = []            # todas las frases de todos los clips, en orden de grabación
for ci, c in enumerate(clips):
    c['phrases'] = []
    if not c['has_audio']:
        continue
    wav = f"{WORK}/audio/clip{ci:02d}.wav"
    run(['ffmpeg', '-y', '-v', 'error', '-i', c['proxy'], '-vn', '-ac', '1', '-ar', '16000', wav])
    try:
        words = transcribe(wav)
    except RuntimeError as e:
        print("Falló Whisper en GPU, reintento en CPU:", str(e)[:200])
        wm = load_whisper(False); words = transcribe(wav)
    for k, ph in enumerate(split_phrases(words)):
        p = {'id': f"c{ci}p{k}", 'clip': ci, 'start': ph[0][0], 'end': ph[-1][1],
             'text': ' '.join(w[2] for w in ph), 'words': ph}
        c['phrases'].append(p); PHRASES.append(p)
    print(f"  {c['name']}: {len(c['phrases'])} frases · «{' '.join(w[2] for w in words)[:110]}…»")

del wm; gc.collect()                       # libera la GPU para el modelo de IA
talk_s = sum(p['end'] - p['start'] for p in PHRASES)
print(f"\n{len(PHRASES)} frases, {talk_s:.0f} s de voz en total")
if not PHRASES:
    raise RuntimeError("No se detectó voz en los clips. Revisa IDIOMA o el audio de los videos.")
open(f"{WORK}/transcripcion.txt", 'w').write("\n".join(f"{p['id']} [{clips[p['clip']]['name']} {p['start']:.1f}s] {p['text']}" for p in PHRASES))

In [ ]:
#@title 5 · IA editora: arma el guion de la promo con tus mejores frases y elige los textos para la pared
# Un fotograma por clip para que la IA vea dónde estás y qué muestras
thumbs = []
for ci, c in enumerate(clips):
    cap = cv2.VideoCapture(c['proxy']); cap.set(cv2.CAP_PROP_POS_FRAMES, c['nframes'] // 2)
    ok, fr = cap.read(); cap.release()
    if not ok: continue
    k = 640 / max(fr.shape[:2]); fr = cv2.resize(fr, None, fx=k, fy=k)
    ok, buf = cv2.imencode('.jpg', fr, [cv2.IMWRITE_JPEG_QUALITY, 80])
    thumbs.append((ci, base64.standard_b64encode(buf.tobytes()).decode()))

listing = "\n".join(f"{p['id']} | clip {p['clip']} | {p['end'] - p['start']:.1f}s | {p['text']}" for p in PHRASES)
PROMPT = f"""Eres editor de videos de TikTok. Arriba hay un fotograma de cada clip. Abajo está la transcripción de lo
que dice el creador en sus clips, dividida en frases con id. Es la promoción de su servidor «{NOMBRE_SERVER}».

Arma el guion de un TikTok de unos {DURACION_OBJETIVO_S} segundos (entre {int(DURACION_OBJETIVO_S * 0.8)} y {int(DURACION_OBJETIVO_S * 1.15)} s sumando las duraciones)
que explique con claridad: qué es el server, qué se puede hacer ahí / por qué vale la pena, y cómo entrar.
Reglas:
- Usa SOLO frases de la lista, con su id exacto. Puedes reordenarlas para que se entienda mejor.
- Empieza con la frase más llamativa como gancho (los primeros 3 segundos deciden si la gente se queda).
- Quita repeticiones, frases cortadas, muletillas y tomas falsas (si una idea se dijo dos veces, deja la mejor toma).
- 'texto_pared': de 1 a 3 palabras clave en MAYÚSCULAS que aparecerán escritas en la pared detrás de él mientras
  dice esa frase (ej. "MODO SUPERVIVENCIA", "SIN LAG", "GRATIS"). Ponlo en más o menos 1 de cada 3 frases, en las
  que dicen algo importante; en las demás deja "". Nada de emojis.
- 'gancho': título corto (máx. 4 palabras) para la pared en el primer plano.
- 'cta': invitación final corta (máx. 6 palabras). Si en la transcripción se menciona IP, Discord o cómo entrar, úsalo.
- 'resumen': 1-2 frases sobre qué explica el video (para las notas del editor).

Transcripción:
{listing}"""

SCHEMA = {'type': 'object', 'additionalProperties': False,
          'required': ['gancho', 'cta', 'resumen', 'guion'],
          'properties': {'gancho': {'type': 'string'}, 'cta': {'type': 'string'}, 'resumen': {'type': 'string'},
                         'guion': {'type': 'array', 'items': {'type': 'object', 'additionalProperties': False,
                                   'required': ['id', 'texto_pared'],
                                   'properties': {'id': {'type': 'string'}, 'texto_pared': {'type': 'string'}}}}}}

def ask_local_qwen(prompt, images):
    import urllib.request
    from openai import OpenAI
    vllm_bin = '/content/vllm_env/bin/vllm'
    if not os.path.exists(vllm_bin):
        raise RuntimeError("vLLM no está instalado (corre la celda 1 en Colab).")
    log = open(f"{WORK}/vllm.log", 'w')
    proc = subprocess.Popen([vllm_bin, 'serve', MODELO_LOCAL, '--port', '8000', '--max-model-len', '32768',
                             '--gpu-memory-utilization', '0.88', '--limit-mm-per-prompt', json.dumps({'image': 32}),
                             '--served-model-name', 'local'], stdout=log, stderr=subprocess.STDOUT)
    print(f"Cargando {MODELO_LOCAL} en la GPU (la 1.ª vez descarga ~56 GB, 5-15 min)…")
    t0 = time.time()
    while True:
        try:
            urllib.request.urlopen('http://localhost:8000/v1/models', timeout=2); break
        except Exception:
            if proc.poll() is not None:
                raise RuntimeError("vLLM se detuvo:\n" + open(f"{WORK}/vllm.log").read()[-3000:])
            if time.time() - t0 > 40 * 60:
                proc.terminate(); raise RuntimeError("vLLM tardó demasiado en arrancar")
            time.sleep(10)
    print(f"Modelo listo en {(time.time() - t0) / 60:.1f} min")
    try:
        client = OpenAI(base_url='http://localhost:8000/v1', api_key='local')
        content = []
        for ci, b in images:
            content += [{'type': 'text', 'text': f"Clip {ci}"},
                        {'type': 'image_url', 'image_url': {'url': f"data:image/jpeg;base64,{b}"}}]
        content.append({'type': 'text', 'text': prompt})
        r = client.chat.completions.create(
            model='local', messages=[{'role': 'user', 'content': content}], max_tokens=6000,
            temperature=0.7, top_p=0.8, presence_penalty=1.5,
            response_format={'type': 'json_schema', 'json_schema': {'name': 'guion', 'schema': SCHEMA}},
            extra_body={'top_k': 20, 'chat_template_kwargs': {'enable_thinking': False}})
        txt = r.choices[0].message.content.strip()
        return json.loads(txt[txt.find('{'): txt.rfind('}') + 1])
    finally:
        proc.terminate()
        try: proc.wait(timeout=60)
        except Exception: proc.kill()

def ask_claude(prompt, images):
    try:
        from google.colab import userdata
        key = userdata.get('ANTHROPIC_API_KEY')
    except Exception:
        key = os.environ.get('ANTHROPIC_API_KEY')
    if not key:
        raise RuntimeError("Falta el secreto ANTHROPIC_API_KEY")
    import anthropic
    client = anthropic.Anthropic(api_key=key)
    content = []
    for ci, b in images:
        content += [{'type': 'text', 'text': f"Clip {ci}"},
                    {'type': 'image', 'source': {'type': 'base64', 'media_type': 'image/jpeg', 'data': b}}]
    content.append({'type': 'text', 'text': prompt})
    with client.beta.messages.stream(
        model='claude-opus-5-5', max_tokens=16000,
        betas=['server-side-fallback-2026-07-01'], fallbacks='default',
        output_config={'effort': 'high', 'format': {'type': 'json_schema', 'schema': SCHEMA}},
        messages=[{'role': 'user', 'content': content}],
    ) as stream:
        msg = stream.get_final_message()
    if msg.stop_reason == 'refusal':
        raise RuntimeError("Claude declinó la petición")
    return json.loads(next(b.text for b in msg.content if b.type == 'text'))

STOP = set("""que de la el en y a los las un una por con para es no se lo me mi te tu su al del como pero más
muy ya este esta eso esto hay son está están ser hacer tiene tienen todo todos ahí aquí entonces bueno pues o si
the and to of is it in you for on this that with are""".split())
def simple_plan():
    """Sin IA: orden de grabación, frases de 3+ palabras, hasta la duración objetivo."""
    plan, tot = [], 0.0
    for k, p in enumerate(PHRASES):
        if len(p['words']) < 3: continue
        words = [re.sub(r'[^\wáéíóúñü]', '', w[2].lower()) for w in p['words']]
        kw = max((w for w in words if w not in STOP), key=len, default='')
        plan.append({'id': p['id'], 'texto_pared': kw.upper() if (len(plan) % 3 == 1 and len(kw) >= 5) else ''})
        tot += p['end'] - p['start']
        if tot >= DURACION_OBJETIVO_S: break
    return {'gancho': NOMBRE_SERVER.upper(), 'cta': '¡Te esperamos en el server!',
            'resumen': 'Guion automático en orden de grabación (sin IA).', 'guion': plan}

PLAN = None
try:
    if MOTOR_IA == 'local_qwen': PLAN = ask_local_qwen(PROMPT, thumbs)
    elif MOTOR_IA == 'claude': PLAN = ask_claude(PROMPT, thumbs)
except Exception as e:
    print(f"⚠ {MOTOR_IA} falló, uso el guion automático:", str(e)[-1500:])

# Validar: ids reales, sin duplicados, y ajustar la duración
byid = {p['id']: p for p in PHRASES}
if PLAN:
    seen = set()
    PLAN['guion'] = [g for g in PLAN['guion'] if g['id'] in byid and not (g['id'] in seen or seen.add(g['id']))]
if not PLAN or not PLAN['guion']:
    PLAN = simple_plan()
while len(PLAN['guion']) > 3 and sum(byid[g['id']]['end'] - byid[g['id']]['start'] for g in PLAN['guion']) > DURACION_OBJETIVO_S * 1.25:
    PLAN['guion'].pop(-2)                    # recorta del medio-final, conserva el cierre
if TEXTO_CTA.strip():
    PLAN['cta'] = TEXTO_CTA.strip()

dur = sum(byid[g['id']]['end'] - byid[g['id']]['start'] for g in PLAN['guion'])
print(f"Gancho: {PLAN['gancho']}  ·  CTA: {PLAN['cta']}  ·  {len(PLAN['guion'])} frases, ~{dur:.0f}s de voz\n")
for g in PLAN['guion']:
    print(f"  {('🧱 ' + g['texto_pared']) if g['texto_pared'] else '':<26} {byid[g['id']]['text']}")

In [ ]:
#@title 6 · Línea de tiempo: cortes en tus frases, voz limpia y música de fondo que baja cuando hablas
HOOK_S, CTA_S = 2.8, 3.0
PIECES = []
for g in PLAN['guion']:
    p = byid[g['id']]; c = clips[p['clip']]
    s0 = max(0.0, p['start'] - 0.10); s1 = min(c['duration'], p['end'] + 0.18)
    prev = PIECES[-1] if PIECES else None
    if prev and prev['clip'] == p['clip'] and 0 <= s0 - prev['s1'] < 0.35:      # frases seguidas: un solo plano
        prev['s1'] = s1; prev['phr'].append((p, g['texto_pared']))
    else:
        PIECES.append({'clip': p['clip'], 's0': s0, 's1': s1, 'phr': [(p, g['texto_pared'])]})

t = 0
WORDS, WALLS = [], []          # palabras retemporizadas y textos de pared en tiempo de salida (frames)
for k, pc in enumerate(PIECES):
    pc['idx'] = k
    pc['src_f0'] = int(round(pc['s0'] * FPS)); pc['n'] = max(1, int(round((pc['s1'] - pc['s0']) * FPS)))
    pc['f0'] = t; pc['f1'] = t + pc['n']; t = pc['f1']
    # punch-in alterno en cortes del mismo clip (disimula el salto)
    same = k > 0 and PIECES[k - 1]['clip'] == pc['clip']
    pc['zoom'] = (1.14 if PIECES[k - 1].get('zoom', 1.0) < 1.07 else 1.0) if same else 1.0
    for p, wall in pc['phr']:
        off = pc['f0'] / FPS - pc['s0']
        WORDS += [(a + off, b + off, w) for a, b, w in p['words']]
        if wall.strip():
            WALLS.append({'f0': int((p['start'] + off) * FPS), 'f1': int((p['end'] + off) * FPS), 'text': wall.strip().upper(), 'piece': k})
TOTAL_FRAMES = t; L = TOTAL_FRAMES / FPS

# Gancho al inicio y CTA al final también van "en la pared"
WALLS = [w for w in WALLS if w['f0'] >= HOOK_S * FPS and w['f1'] <= TOTAL_FRAMES - CTA_S * FPS]
WALLS.insert(0, {'f0': 0, 'f1': int(min(HOOK_S, PIECES[0]['n'] / FPS) * FPS), 'text': PLAN['gancho'].upper(), 'piece': 0, 'big': True})
WALLS.append({'f0': TOTAL_FRAMES - int(CTA_S * FPS), 'f1': TOTAL_FRAMES, 'text': PLAN['cta'].upper(), 'piece': len(PIECES) - 1, 'big': True})
for w in WALLS:                       # cada texto se queda dentro de su plano (no cruza cortes)
    pc = PIECES[w['piece']]; w['f0'] = max(w['f0'], pc['f0']); w['f1'] = min(max(w['f1'], w['f0'] + 36), pc['f1'])
WALLS = [w for w in WALLS if w['f1'] - w['f0'] >= 12]
print(f"{len(PIECES)} planos · {L:.1f}s · {len(WALLS)} textos en la pared · {len(WORDS)} palabras subtituladas")

# ---------- Audio: voz por plano + limpieza + música con ducking ----------
parts = []
for pc in PIECES:
    c = clips[pc['clip']]; d = pc['n'] / FPS; p = f"{WORK}/audio/piece{pc['idx']:03d}.wav"
    src = ['-f', 'lavfi', '-i', 'anullsrc=r=48000:cl=stereo'] if not c['has_audio'] else ['-ss', f"{pc['src_f0'] / FPS:.3f}", '-i', c['proxy']]
    run(['ffmpeg', '-y', '-v', 'error', *src, '-vn', '-ac', '2', '-ar', '48000',
         '-af', f"apad,atrim=0:{d:.4f},afade=t=in:d=0.02,afade=t=out:st={max(0, d - 0.03):.4f}:d=0.03", p])
    parts.append(p)
open(f"{WORK}/alist.txt", 'w').writelines(f"file '{p}'\n" for p in parts)
VOICE_WAV = f"{WORK}/voz.wav"
run(['ffmpeg', '-y', '-v', 'error', '-f', 'concat', '-safe', '0', '-i', f"{WORK}/alist.txt", '-c', 'copy', VOICE_WAV])

MIX_WAV = f"{WORK}/mix.wav"
music = os.path.join(FOLDER, SOUNDTRACK)
run(['ffmpeg', '-y', '-v', 'error', '-i', VOICE_WAV, '-stream_loop', '-1', '-i', music, '-filter_complex',
     "[0:a]highpass=f=80,afftdn=nf=-25,acompressor=threshold=-20dB:ratio=3:attack=5:release=80,"
     "loudnorm=I=-14:TP=-1.5:LRA=7,asplit=2[v1][v2];"
     f"[1:a]atrim=0:{L:.3f},asetpts=N/SR/TB,volume={VOLUMEN_MUSICA},afade=t=in:d=0.5,"
     f"afade=t=out:st={max(0, L - 1.5):.3f}:d=1.5[m];"
     "[m][v2]sidechaincompress=threshold=0.03:ratio=8:attack=20:release=350[md];"
     f"[v1][md]amix=inputs=2:normalize=0:duration=first,alimiter=limit=0.95,apad,atrim=0:{L:.4f}[o]",
     '-map', '[o]', '-ar', '48000', '-t', f"{L:.3f}", MIX_WAV])
print("Audio listo:", MIX_WAV)

In [ ]:
#@title 7 · Look: color, encuadre vertical en tu cara, textos pegados a la pared (detrás de ti) y subtítulos TikTok
from PIL import Image, ImageDraw, ImageFont, ImageFilter
import urllib.request
FONT_DIR = f"{WORK}/fonts"; os.makedirs(FONT_DIR, exist_ok=True)
FONT_URLS = {'BebasNeue-Regular.ttf': 'https://github.com/google/fonts/raw/main/ofl/bebasneue/BebasNeue-Regular.ttf',
             'Anton-Regular.ttf': 'https://github.com/google/fonts/raw/main/ofl/anton/Anton-Regular.ttf'}
for fn, url in FONT_URLS.items():
    p = os.path.join(FONT_DIR, fn)
    if not os.path.exists(p):
        try: urllib.request.urlretrieve(url, p)
        except Exception as e: print("No pude bajar", fn, e)
def font(fn, size):
    for cand in (os.path.join(FONT_DIR, fn), '/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf'):
        try: return ImageFont.truetype(cand, size)
        except Exception: pass
    return ImageFont.load_default(size=size)

# ---------- Color ----------
def curve(points):
    xs, ys = zip(*points); return np.clip(np.interp(np.arange(256), xs, ys), 0, 255).astype(np.uint8)
GRADES = {'natural': (curve([(0, 8), (128, 124), (255, 248)]), curve([(0, 6), (128, 128), (255, 252)]),
                      curve([(0, 6), (128, 133), (255, 255)]), 1.08),
          'teal_orange': (curve([(0, 22), (64, 74), (128, 126), (192, 170), (255, 236)]),
                          curve([(0, 10), (64, 60), (128, 128), (192, 196), (255, 248)]),
                          curve([(0, 6), (64, 52), (128, 134), (192, 208), (255, 255)]), 1.12),
          'warm_film': (curve([(0, 18), (128, 116), (255, 228)]), curve([(0, 14), (128, 128), (255, 244)]),
                        curve([(0, 16), (128, 140), (255, 252)]), 0.95)}
def grade(fr):
    if GRADE == 'ninguno': return fr
    LB, LG, LR, SAT = GRADES[GRADE]
    b, g, r = cv2.split(fr); fr = cv2.merge([cv2.LUT(b, LB), cv2.LUT(g, LG), cv2.LUT(r, LR)])
    if SAT != 1.0:
        hsv = cv2.cvtColor(fr, cv2.COLOR_BGR2HSV); hsv[..., 1] = cv2.convertScaleAbs(hsv[..., 1], alpha=SAT)
        fr = cv2.cvtColor(hsv, cv2.COLOR_HSV2BGR)
    return fr

# ---------- Persona: detección (encuadre) y segmentación (texto detrás de ti) ----------
from ultralytics import YOLO
seg_model = YOLO('yolo11l-seg.pt')
def person_mask(fr):
    r = seg_model.predict(fr, classes=[0], conf=0.25, verbose=False, retina_masks=True, imgsz=640)[0]
    if r.masks is None or not len(r.masks):
        return np.zeros(fr.shape[:2], np.float32)
    m = r.masks.data.cpu().numpy().max(0).astype(np.float32)
    if m.shape != fr.shape[:2]: m = cv2.resize(m, (fr.shape[1], fr.shape[0]))
    m = cv2.dilate(m, np.ones((7, 7), np.uint8))
    return cv2.GaussianBlur(m, (0, 0), 3)

def track_centers(pc):
    """Centro X (0-1) de la persona por frame del plano, suavizado como un operador de cámara."""
    c = clips[pc['clip']]; n = pc['n']
    cap = cv2.VideoCapture(c['proxy']); cap.set(cv2.CAP_PROP_POS_FRAMES, pc['src_f0'])
    xs, ts = [], []
    for i in range(n):
        ok, fr = cap.read()
        if not ok: break
        if i % 4 == 0:
            r = seg_model.predict(fr, classes=[0], conf=0.3, verbose=False, imgsz=640)[0]
            if len(r.boxes):
                b = r.boxes.xyxy.cpu().numpy(); a = (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1])
                j = int(a.argmax()); xs.append(float((b[j, 0] + b[j, 2]) / 2 / fr.shape[1])); ts.append(i)
    cap.release()
    if not xs: return np.full(n, 0.5)
    raw = np.interp(np.arange(n), ts, xs); out = np.empty(n); out[0] = np.median(raw[:15])
    for i in range(1, n): out[i] = out[i - 1] * 0.94 + raw[i] * 0.06
    return out

def crop_frame(fr, cxn, zoom):
    h, w = fr.shape[:2]; tgt = OUT_W / OUT_H
    cw, ch = (h * tgt, h) if w / h > tgt else (w, w / tgt)
    cw, ch = cw / zoom, ch / zoom
    x0 = np.clip(cxn * w - cw / 2, 0, w - cw); y0 = np.clip((h - ch) * 0.35, 0, h - ch)   # un poco de aire arriba
    M = np.float32([[OUT_W / cw, 0, -x0 * OUT_W / cw], [0, OUT_H / ch, -y0 * OUT_H / ch]])
    return cv2.warpAffine(fr, M, (OUT_W, OUT_H), flags=cv2.INTER_CUBIC)

for pc in PIECES:
    c = clips[pc['clip']]
    pc['cx'] = track_centers(pc) if c['W'] / c['H'] > OUT_W / OUT_H else np.full(pc['n'], 0.5)

# ---------- Texto pegado a la pared ----------
def wall_text_rgba(text, big):
    words = text.split(); lines, cur = [], ''
    for w in words:                              # máx. ~11 caracteres por línea
        if cur and len(cur) + 1 + len(w) > 11: lines.append(cur); cur = w
        else: cur = (cur + ' ' + w).strip()
    lines.append(cur)
    size = 230 if big else 190
    f = font('BebasNeue-Regular.ttf', size)
    W = int(max(f.getlength(l) for l in lines)) + 40; lh = int(size * 0.92); H = lh * len(lines) + 40
    img = Image.new('RGBA', (W, H), (0, 0, 0, 0)); d = ImageDraw.Draw(img)
    for k, l in enumerate(lines):
        d.text(((W - f.getlength(l)) / 2, 20 + k * lh), l, font=f, fill=(246, 240, 228, 255))
    return np.array(img)

def plane_quad(w, h, cx, cy, yaw, pitch, scale, focal=1600):
    pts = np.array([[-w / 2, -h / 2, 0], [w / 2, -h / 2, 0], [w / 2, h / 2, 0], [-w / 2, h / 2, 0]], np.float32) * scale
    cy_, sy = math.cos(math.radians(yaw)), math.sin(math.radians(yaw))
    cp, sp = math.cos(math.radians(pitch)), math.sin(math.radians(pitch))
    R = np.array([[cy_, 0, sy], [0, 1, 0], [-sy, 0, cy_]]) @ np.array([[1, 0, 0], [0, cp, -sp], [0, sp, cp]])
    P = pts @ R.T; z = P[:, 2] + focal
    return np.stack([cx + P[:, 0] * focal / z, cy + P[:, 1] * focal / z], 1).astype(np.float32)

class WallText:
    """Texto en un plano de la pared: se coloca donde hay pared libre, sigue el movimiento de cámara
    (homografía de puntos del fondo) y queda detrás de la persona."""
    def __init__(self, spec, frame, mask):
        self.spec = spec; self.rgba = wall_text_rgba(spec['text'], spec.get('big'))
        h, w = self.rgba.shape[:2]
        bg = 1 - mask
        scale = min(0.70 * OUT_W / w, 0.17 * OUT_H / h)
        best = None
        for cx in (0.5, 0.33, 0.67):                     # dónde se ve más pared libre
            for cy in (0.15, 0.23, 0.31, 0.42):
                yaw = {0.5: random.choice([-18, 18]), 0.33: 30, 0.67: -30}[cx]
                q = self._fit(plane_quad(w, h, OUT_W * cx, OUT_H * cy, yaw, -3, scale))
                poly = np.zeros((OUT_H // 4, OUT_W // 4), np.uint8)
                cv2.fillConvexPoly(poly, (q / 4).astype(np.int32), 1)
                vis = bg[::4, ::4][poly > 0].mean() if poly.any() else 0
                score = vis + (0.05 if cx == 0.5 else 0) - 0.06 * abs(cy - 0.23) / 0.2
                if best is None or score > best[0]: best = (score, q, vis)
        _, self.quad0, vis = best
        # Hay pared a la vista y se detectó a la persona -> texto en la pared, detrás de ti.
        # Si no (primerísimo plano de la cara), va arriba, delante, con sombra para que se lea.
        self.occlude = vis >= 0.35 and mask.mean() > 0.03
        if not self.occlude:
            self.quad0 = self._fit(plane_quad(w, h, OUT_W / 2, OUT_H * 0.11, random.choice([-12, 12]), -4, scale * 0.8))
        self.H = np.eye(3); self.prev = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY); self.pts = self._features(self.prev, mask)

    @staticmethod
    def _fit(q):
        """Mete el texto completo en pantalla (márgenes seguros de TikTok)."""
        x0, x1 = q[:, 0].min(), q[:, 0].max()
        lo, hi = OUT_W * 0.05, OUT_W * 0.95
        if x1 - x0 > hi - lo:
            c = q.mean(0); q = (q - c) * ((hi - lo) / (x1 - x0)) + c; x0, x1 = q[:, 0].min(), q[:, 0].max()
        q = q.copy(); q[:, 0] += max(0, lo - x0) - max(0, x1 - hi)
        q[:, 1] += max(0, OUT_H * 0.08 - q[:, 1].min())
        return q

    def _features(self, gray, mask):
        m = ((1 - mask) > 0.5).astype(np.uint8) * 255
        return cv2.goodFeaturesToTrack(gray, 300, 0.01, 12, mask=m)

    def update(self, frame, mask):
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        if self.pts is not None and len(self.pts) >= 8:
            nxt, st, _ = cv2.calcOpticalFlowPyrLK(self.prev, gray, self.pts, None, winSize=(21, 21), maxLevel=3)
            good0, good1 = self.pts[st[:, 0] == 1], nxt[st[:, 0] == 1]
            if len(good0) >= 8:
                Hs, inl = cv2.findHomography(good0, good1, cv2.RANSAC, 3.0)
                if Hs is not None and abs(np.linalg.det(Hs[:2, :2]) - 1) < 0.2:
                    Hn = Hs @ self.H
                    c0 = self.quad0.mean(0); c1 = cv2.perspectiveTransform(c0.reshape(1, 1, 2), Hn)[0, 0]
                    if np.abs(c1 - c0).max() < OUT_W * 0.25:      # evita derivas grandes
                        self.H = Hn
                self.pts = good1.reshape(-1, 1, 2)
        if self.pts is None or len(self.pts) < 60:
            self.pts = self._features(gray, mask)
        self.prev = gray

    def draw(self, frame, mask, t, T):
        quad = self._fit(cv2.perspectiveTransform(self.quad0.reshape(-1, 1, 2), self.H).reshape(4, 2))
        h, w = self.rgba.shape[:2]
        rgba = self.rgba.copy()
        reveal = min(1.0, t / 0.4)                       # se "escribe" de izquierda a derecha
        if reveal < 1:
            ramp = np.clip((np.linspace(0, 1, w) - (reveal * 1.25 - 0.25)) / -0.25, 0, 1)
            rgba[..., 3] = (rgba[..., 3] * ramp[None, :]).astype(np.uint8)
        fade = min(1.0, (T - t) / 0.25)
        M = cv2.getPerspectiveTransform(np.float32([[0, 0], [w, 0], [w, h], [0, h]]), quad.astype(np.float32))
        warped = cv2.warpPerspective(rgba, M, (OUT_W, OUT_H), flags=cv2.INTER_LINEAR, borderValue=(0, 0, 0, 0))
        warped = cv2.GaussianBlur(warped, (0, 0), 0.8)    # misma suavidad que la cámara
        a = warped[..., 3:4].astype(np.float32) / 255 * fade * 0.93
        if self.occlude: a *= (1 - mask)[..., None]      # la persona tapa el texto: está en la pared
        # iluminación de la pared: el texto toma la luz/sombra del fondo
        lum = cv2.GaussianBlur(cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY).astype(np.float32), (0, 0), 25)
        if self.occlude:
            shade = np.clip(lum / (np.percentile(lum, 90) + 1e-6), 0.45, 1.05)[..., None]
            col = warped[..., [2, 1, 0]].astype(np.float32) * shade
            sh = np.roll(a, (6, 5), (0, 1)) * 0.28       # sombra de contacto suave (pintado en la pared)
        else:
            col = warped[..., [2, 1, 0]].astype(np.float32)
            sh = cv2.GaussianBlur(np.roll(a, (8, 6), (0, 1)), (0, 0), 6)[..., None] * 0.6
        out = frame.astype(np.float32) * (1 - sh)
        frame[:] = np.clip(out * (1 - a) + col * a, 0, 255).astype(np.uint8)

# ---------- Subtítulos estilo TikTok (palabra activa resaltada) ----------
def ass_time(t):
    t = max(0, t); return f"{int(t // 3600)}:{int(t % 3600 // 60):02d}:{t % 60:05.2f}"
def write_ass(path):
    chunks, cur = [], []
    for w in WORDS:
        if cur and (len(cur) >= 3 or w[0] - cur[-1][1] > 0.5): chunks.append(cur); cur = []
        cur.append(w)
    if cur: chunks.append(cur)
    L_ = ["[Script Info]", "ScriptType: v4.00+", f"PlayResX: {OUT_W}", f"PlayResY: {OUT_H}", "WrapStyle: 0", "",
          "[V4+ Styles]",
          "Format: Name, Fontname, Fontsize, PrimaryColour, SecondaryColour, OutlineColour, BackColour, Bold, Italic, Underline, StrikeOut, ScaleX, ScaleY, Spacing, Angle, BorderStyle, Outline, Shadow, Alignment, MarginL, MarginR, MarginV, Encoding",
          "Style: TT,Anton,96,&H00FFFFFF,&H00FFFFFF,&H00000000,&H80000000,0,0,0,0,100,100,1,0,1,7,3,2,90,90,560,1",
          "", "[Events]", "Format: Layer, Start, End, Style, Name, MarginL, MarginR, MarginV, Effect, Text"]
    clean = lambda s: re.sub(r'[{}\\]', '', s).upper().strip(' ,')
    for ch in chunks:
        for i, (a, b, _) in enumerate(ch):
            end = ch[i + 1][0] if i + 1 < len(ch) else max(b, a + 0.3)
            txt = ' '.join((r"{\c&H00F5FF&\fscx112\fscy112}" + clean(w) + r"{\r}") if j == i else clean(w)
                           for j, (_, _, w) in enumerate(ch))
            pop = r"{\fad(40,0)}" if i == 0 else ""
            L_.append(f"Dialogue: 0,{ass_time(a)},{ass_time(end)},TT,,0,0,0,,{pop}{txt}")
    open(path, 'w', encoding='utf-8').write("\n".join(L_) + "\n")
ASS = f"{WORK}/subs_tiktok.ass"; write_ass(ASS)
print("Look listo.")

In [ ]:
#@title 8 · Render final 9:16 (1080×1920) para TikTok / Reels / Shorts
OUT = os.path.join(WORK, f"{re.sub(r'[^A-Za-z0-9]+', '_', NOMBRE_SERVER).strip('_').lower()}_promo_tiktok.mp4")
vf = f"ass={os.path.basename(ASS)}:fontsdir={FONT_DIR}" if WORDS else "null"
cmd = ['ffmpeg', '-y', '-v', 'error', '-f', 'rawvideo', '-pix_fmt', 'bgr24', '-s', f'{OUT_W}x{OUT_H}', '-r', str(FPS),
       '-i', '-', '-i', MIX_WAV, '-vf', vf, '-map', '0:v', '-map', '1:a', *VENC, '-pix_fmt', 'yuv420p',
       '-color_primaries', 'bt709', '-color_trc', 'bt709', '-colorspace', 'bt709',
       '-c:a', 'aac', '-b:a', '256k', '-t', f"{L:.4f}", '-movflags', '+faststart', OUT]
pipe = subprocess.Popen(cmd, stdin=subprocess.PIPE, stderr=subprocess.PIPE, cwd=WORK)
t0 = time.time(); g = 0; active = {}
for pc in PIECES:
    c = clips[pc['clip']]; n = pc['n']
    cap = cv2.VideoCapture(c['proxy']); cap.set(cv2.CAP_PROP_POS_FRAMES, pc['src_f0']); last = None
    walls_here = [w for w in WALLS if w['piece'] == pc['idx']]
    for i in range(n):
        ok, fr = cap.read()
        fr = fr if ok else (last if last is not None else np.zeros((c['H'], c['W'], 3), np.uint8)); last = fr
        z = pc['zoom'] * (1 + 0.03 * i / max(1, n - 1))            # empuje lento dentro del plano
        f = grade(crop_frame(fr, pc['cx'][min(i, len(pc['cx']) - 1)], z))
        now = [w for w in walls_here if w['f0'] <= g < w['f1']]
        if now:
            mask = person_mask(f)
            for w in now:
                key = id(w)
                if key not in active:
                    active[key] = WallText(w, f, mask)
                else:
                    active[key].update(f, mask)
                active[key].draw(f, mask, (g - w['f0']) / FPS, (w['f1'] - w['f0']) / FPS)
        for w in walls_here:
            if g >= w['f1']: active.pop(id(w), None)
        try:
            pipe.stdin.write(f.tobytes())
        except BrokenPipeError:
            raise RuntimeError("ffmpeg se detuvo: " + pipe.stderr.read().decode()[-2000:])
        g += 1
    cap.release()
    print(f"\r  {g}/{TOTAL_FRAMES} frames ({g / max(1e-6, time.time() - t0):.1f} fps)", end='')
pipe.stdin.close(); err = pipe.stderr.read().decode(); pipe.wait()
if pipe.returncode != 0:
    raise RuntimeError(err[-2000:])
print(f"\n✓ {OUT}")

In [ ]:
#@title 9 · Guardar en Drive (carpeta Edicion_IA) + guion y notas
notes = [f"# {NOMBRE_SERVER} — promo TikTok", "", PLAN.get('resumen', ''), "",
         f"- Duración: {L:.1f}s · {len(PIECES)} planos · gancho: **{PLAN['gancho']}** · CTA: **{PLAN['cta']}**", "",
         "| # | Clip | Texto en la pared | Lo que dices |", "|---|---|---|---|"]
for k, g_ in enumerate(PLAN['guion']):
    p = byid[g_['id']]
    notes.append(f"| {k + 1} | {clips[p['clip']]['name']} @ {p['start']:.1f}s | {g_['texto_pared'] or '—'} | {p['text']} |")
open(f"{WORK}/guion_promo_tiktok.md", 'w', encoding='utf-8').write("\n".join(notes))
for p in (OUT, f"{WORK}/guion_promo_tiktok.md", f"{WORK}/transcripcion.txt"):
    shutil.copy(p, OUT_DIR); print("→", os.path.join(OUT_DIR, os.path.basename(p)))
if IN_COLAB:
    from IPython.display import Video, display
    prev = f"{WORK}/preview.mp4"
    run(['ffmpeg', '-y', '-v', 'error', '-i', OUT, '-vf', 'scale=360:-2', '-c:v', 'libx264', '-crf', '30', '-c:a', 'aac', '-b:a', '96k', prev])
    display(Video(prev, embed=True, width=300))
print("\n¿Ajustes? Edita PLAN['guion'] (orden, 'texto_pared'), PLAN['gancho'] / PLAN['cta'] y vuelve a correr las celdas 6–9.")